In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import h5py
import os
DATA_DIR = "/kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final"
SUBFOLDER = "t15.2025.03.14"
test_file_path = os.path.join(DATA_DIR, SUBFOLDER, "data_test.hdf5")
print(f"Inspecting test file: {test_file_path}")
try:
    with h5py.File(test_file_path, "r") as f:
        first_trial_key = sorted(list(f.keys()))[0]
        print(f"Inspecting keys inside: {first_trial_key}")
        trial_group = f[first_trial_key]
        print(f"Keys found: {list(trial_group.keys())}")
        for key in trial_group.keys():
            print(f"  • {key}: shape {trial_group[key].shape}, dtype {trial_group[key].dtype}")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
!pip install jiwer

In [ ]:
print("Importing libraries...")
import os
import yaml
import h5py
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
class CFG:
    N_HEAD = 8 
    EPOCHS = 5
    LR = 1e-3
    BATCH_SIZE = 32
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DATA_DIR = "/kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final"
    CHECKPOINT_PATH = "/kaggle/input/brain-to-text-25/t15_pretrained_rnn_baseline/t15_pretrained_rnn_baseline/checkpoint/best_checkpoint"
    ARGS_PATH = "/kaggle/input/brain-to-text-25/t15_pretrained_rnn_baseline/t15_pretrained_rnn_baseline/checkpoint/args.yaml"
    COMPETITION_TEST_PATH = "/kaggle/input/brain-to-text-25/data_test.hdf5"
print(f"Running on device: {CFG.DEVICE}")

In [ ]:
with open(CFG.ARGS_PATH, "r") as f:
    args = yaml.safe_load(f)
print("Loaded model config from args.yaml:")
INPUT_SIZE = args.get("input_size", 256)
HIDDEN_SIZE = args.get("hidden_size", 512)
OUTPUT_SIZE = args.get("output_size", 29)
NUM_LAYERS = args.get("num_layers", 1)
print(args)

In [ ]:
print("Inspecting a TRAINING file to find the target key...")
TRAIN_SUBFOLDER = "t15.2025.01.10" 
train_file_path = os.path.join(CFG.DATA_DIR, TRAIN_SUBFOLDER, "data_train.hdf5")
try:
    with h5py.File(train_file_path, "r") as f:
        first_trial_key = sorted(list(f.keys()))[0]
        print(f"Inspecting keys inside trial: {first_trial_key}")
        trial_group = f[first_trial_key]
        print(f"!!! TẤT CẢ CÁC KEY TÌM THẤY: {list(trial_group.keys())} !!!")
        for key in trial_group.keys():
            if key != 'input_features':
                print(f"  • Tìm thấy key khả nghi: '{key}'")
                print(f"    Shape: {trial_group[key].shape}, Dtype: {trial_group[key].dtype}")
                print(f"    Sample data: {trial_group[key][:10]}")
except Exception as e:
    print(f"Lỗi khi kiểm tra: {e}")

In [ ]:
from torch.utils.data import ConcatDataset, Dataset
def temporal_mask(data, mask_percentage=0.05, mask_value=0.0):
    """
    Applies temporal masking to a 2D tensor [Sequence, Features].
    """
    if not torch.is_tensor(data):
        data = torch.tensor(data, dtype=torch.float32)
    seq_len, _ = data.shape
    num_to_mask = int(seq_len * mask_percentage)
    if num_to_mask > 0:
        mask_indices = torch.randperm(seq_len)[:num_to_mask]
        data[mask_indices, :] = mask_value
    return data
class BrainDataset(Dataset):
    """
    Reads data from a single HDF5 file (e.g., data_train.hdf5).
    - input_key: The name of the HDF5 dataset for input features.
    - target_key: The name of the HDF5 dataset for target sequences (indices).
    - is_test: If True, __getitem__ also returns the trial_key.
    """
    def __init__(self, hdf5_file, input_key="input_features", target_key="phoneme_indices", is_test=False, use_augmentation=False):
        self.file_path = hdf5_file
        self.input_key = input_key
        self.target_key = target_key
        self.is_test = is_test
        self.use_augmentation = use_augmentation 
        self.file = None
        try:
            with h5py.File(self.file_path, "r") as f:
                self.trial_keys = sorted(list(f.keys()))
        except FileNotFoundError:
            print(f"Warning: File not found {self.file_path}, creating empty dataset.")
            self.trial_keys = []
    def __len__(self):
        return len(self.trial_keys)
    def __getitem__(self, idx):
        if self.file is None:
            self.file = h5py.File(self.file_path, "r")
        trial_key = self.trial_keys[idx]
        trial_group = self.file[trial_key]
        x_data = trial_group[self.input_key][:]
        x = torch.tensor(x_data, dtype=torch.float32)
        if self.use_augmentation and not self.is_test:
            x = temporal_mask(x, mask_percentage=0.1)
        if self.target_key in trial_group:
            y_data = trial_group[self.target_key][:]
            y = torch.tensor(y_data, dtype=torch.long)
        else:
            y = torch.tensor([], dtype=torch.long)
        if self.is_test:
            return x, y, trial_key
        else:
            return x, y
def load_datasets():
    """
    Scans all subfolders in CFG.DATA_DIR and creates combined
    train, val, and test datasets from all found files.
    """
    train_datasets = []
    val_datasets = []
    test_datasets = []
    subfolders = [f.path for f in os.scandir(CFG.DATA_DIR) if f.is_dir()]
    print(f"Found {len(subfolders)} session folders.")
    for subfolder_path in subfolders:
        session_name = os.path.basename(subfolder_path)
        train_file = os.path.join(subfolder_path, "data_train.hdf5")
        val_file = os.path.join(subfolder_path, "data_val.hdf5")
        test_file = os.path.join(subfolder_path, "data_test.hdf5")
        train_set = BrainDataset(train_file, input_key="input_features", target_key="seq_class_ids", is_test=False, use_augmentation=True)
        val_set = BrainDataset(val_file, input_key="input_features", target_key="seq_class_ids", is_test=False, use_augmentation=False)
        test_set = BrainDataset(test_file, input_key="input_features", target_key="seq_class_ids", is_test=True, use_augmentation=False) 
        if len(train_set) > 0:
            train_datasets.append(train_set)
        if len(val_set) > 0:
            val_datasets.append(val_set)
        if len(test_set) > 0:
            test_datasets.append(test_set)
    full_train_dataset = ConcatDataset(train_datasets)
    full_val_dataset = ConcatDataset(val_datasets)
    full_test_dataset = ConcatDataset(test_datasets)
    return full_train_dataset, full_val_dataset, full_test_dataset
print("Loading Train/Val/Test data from session folders...")
train_dataset, val_dataset, test_dataset = load_datasets()
print("="*40)
print(f"Total Train samples: {len(train_dataset)}")
print(f"Total Val samples: {len(val_dataset)}")
print(f"Total Test samples: {len(test_dataset)}")
print("="*40)
sample_x_train, sample_y_train = train_dataset[0]
print(f"Train sample X shape: {sample_x_train.shape}")
print(f"Train sample Y (indices): {sample_y_train}")
print(f"Train sample Y shape: {sample_y_train.shape}, dtype: {sample_y_train.dtype}")
if len(test_dataset) > 0:
    sample_x_test, sample_y_test, _ = test_dataset[0]
    print(f"Test sample X shape: {sample_x_test.shape}")
    print(f"Test sample Y (dummy): {sample_y_test}")
    print(f"Test sample Y shape: {sample_y_test.shape}, dtype: {sample_y_test.dtype}")
else:
    print(f"Warning: Competition test file not found at {CFG.COMPETITION_TEST_PATH}")
    print("This is normal. The file will be present during submission.")

In [ ]:
print("🕵️ Inspecting a training file to find the correct label key...")
import h5py
import os
train_file_to_inspect = None
subfolders = [f.path for f in os.scandir(CFG.DATA_DIR) if f.is_dir()]
for subfolder_path in sorted(subfolders):
    train_file = os.path.join(subfolder_path, "data_train.hdf5")
    if os.path.exists(train_file):
        train_file_to_inspect = train_file
        break
if train_file_to_inspect:
    print(f"Inspecting file: {train_file_to_inspect}")
    try:
        with h5py.File(train_file_to_inspect, "r") as f:
            first_trial_key = sorted(list(f.keys()))[0]
            print(f"Inspecting keys inside trial: {first_trial_key}")
            trial_group = f[first_trial_key]
            print(f"\n--- 💡 ALL KEYS FOUND IN THIS TRIAL 💡 ---")
            for key in trial_group.keys():
                print(f"  • {key}")
            print(f"-------------------------------------------\n")
            print("Find the key that looks like labels (e.g., 'targets', 'labels', 'phonemes') and use it in the next step.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Error: Could not find any data_train.hdf5 files to inspect.")

In [ ]:
VOCAB = [
    'AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 
    'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 
    'ZH', '|'
]
OUTPUT_SIZE = len(VOCAB) + 1
BLANK_ID = 0
DATA_INPUT_SIZE = 512
ADAPTER_OUTPUT_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
IS_BIDIRECTIONAL = False
print(f"Vocabulary Config: {len(VOCAB)} phonemes + 1 blank = {OUTPUT_SIZE} classes.")
print(f"Model Config: Data(512) -> Adapter(256) -> RNN(256, {HIDDEN_SIZE}) -> FC({HIDDEN_SIZE}, {OUTPUT_SIZE})")
class RecurrentModel(nn.Module):
    def __init__(self, model_type, data_input_size, adapter_output_size, 
                 hidden_size, output_size, num_layers, bidirectional):
        super().__init__()
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional
        self.adapter_layer = nn.Linear(data_input_size, adapter_output_size)
        rnn_args = {
            'input_size': adapter_output_size,
            'hidden_size': hidden_size,
            'num_layers': num_layers,
            'batch_first': True,
            'bidirectional': bidirectional
        }
        if model_type == "LSTM": self.rnn = nn.LSTM(**rnn_args)
        elif model_type == "GRU": self.rnn = nn.GRU(**rnn_args)
        elif model_type == "RNN": self.rnn = nn.RNN(**rnn_args)
        else: raise ValueError("Invalid model_type")
        fc_in_features = hidden_size * 2 if bidirectional else hidden_size
        self.fc = nn.Linear(fc_in_features, output_size)
    def forward(self, x):
        x = self.adapter_layer(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return nn.functional.log_softmax(out, dim=2)
class TransformerEncModel(nn.Module):
    def __init__(self, data_input_size, adapter_output_size, n_head, num_layers, 
                 dim_feedforward, output_size):
        super().__init__()
        self.adapter_layer = nn.Linear(data_input_size, adapter_output_size)
        self.d_model = adapter_output_size
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model, nhead=n_head, 
            dim_feedforward=dim_feedforward,
            batch_first=True, dropout=0.1
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(self.d_model, output_size)
    def forward(self, x):
        x = self.adapter_layer(x)
        out = self.transformer_encoder(x)
        out = self.fc(out)
        return nn.functional.log_softmax(out, dim=2)

In [ ]:
import torch.nn.utils.rnn as rnn_utils
def custom_collate(batch):
    """
    Custom collate function for CTC (Sequence-to-Sequence).
    Pads both x (inputs) and y (targets) and returns their original lengths.
    `batch` is a list of tuples: (x, y) or (x, y, key)
    """
    is_test = len(batch[0]) == 3
    if is_test:
        xs, ys, keys = zip(*batch)
    else:
        xs, ys = zip(*batch)
    x_lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    y_lengths = torch.tensor([len(y) for y in ys], dtype=torch.long)
    padded_xs = rnn_utils.pad_sequence(xs, batch_first=True, padding_value=0.0)
    padded_ys = rnn_utils.pad_sequence(ys, batch_first=True, padding_value=0)
    if is_test:
        return padded_xs, padded_ys, x_lengths, y_lengths, keys
    else:
        return padded_xs, padded_ys, x_lengths, y_lengths
train_loader = DataLoader(
    train_dataset, 
    batch_size=CFG.BATCH_SIZE, 
    shuffle=True, 
    collate_fn=custom_collate
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=CFG.BATCH_SIZE, 
    shuffle=False, 
    collate_fn=custom_collate 
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=CFG.BATCH_SIZE, 
    shuffle=False, 
    collate_fn=custom_collate
)
print("DataLoaders created with CTC-ready padding.")
try:
    x_batch, y_batch, x_len, y_len = next(iter(train_loader))
    print("\nChecking one batch from train_loader:")
    print(f"  x_batch shape: {x_batch.shape}")
    print(f"  y_batch shape: {y_batch.shape}")
    print(f"  x_lengths shape: {x_len.shape}, sample: {x_len[:5]}")
    print(f"  y_lengths shape: {y_len.shape}, sample: {y_len[:5]}")
except Exception as e:
    print(f"\nCould not get batch from train_loader (is it empty?): {e}")

In [ ]:
import jiwer
import shutil
TOKEN_MAP = {i + 1: phoneme for i, phoneme in enumerate(VOCAB)}
TOKEN_MAP[BLANK_ID] = ""
print("Token map created:")
print(f"  Index 0: '{TOKEN_MAP[0]}' (BLANK)")
print(f"  Index 1: '{TOKEN_MAP[1]}' (e.g., AA)")
print(f"  Index 40: '{TOKEN_MAP[40]}' (e.g., |)")
experiments_to_run = [
    ("RNN", True),
    ("RNN", False),
    ("LSTM", True),
    ("LSTM", False),
    ("GRU", True),
    ("GRU", False),
    ("TRANSFORMER", True),
    ("TRANSFORMER", False),
]
all_experiment_results = {}
def greedy_decoder(logits, token_map):
    pred_indices = torch.argmax(logits, dim=-1)
    collapsed_indices = torch.unique_consecutive(pred_indices)
    final_indices = [idx.item() for idx in collapsed_indices if idx.item() != BLANK_ID]
    phonemes = [token_map.get(i, "?") for i in final_indices]
    text = " ".join(phonemes)
    return text
def decode_true_target(target_indices, token_map):
    phonemes = [token_map.get(i.item(), "?") for i in target_indices]
    text = " ".join(phonemes)
    return text
def train_one_epoch(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for x, y, x_lengths, y_lengths in tqdm(train_loader, desc=f"Epoch {epoch} [Train]", leave=False):
        x, y, x_lengths, y_lengths = x.to(CFG.DEVICE), y.to(CFG.DEVICE), x_lengths.to(CFG.DEVICE), y_lengths.to(CFG.DEVICE)
        optimizer.zero_grad()
        y_pred = model(x)
        y_pred_for_loss = y_pred.permute(1, 0, 2)
        loss = criterion(y_pred_for_loss, y, x_lengths, y_lengths)
        if torch.isinf(loss) or torch.isnan(loss):
            print("Warning: Skipping batch with inf/nan loss")
            continue
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    return running_loss / len(train_loader.dataset)
def validate_one_epoch(epoch, model, val_loader, criterion, token_map):
    model.eval()
    val_loss = 0.0
    all_pred_texts = []
    all_true_texts = []
    with torch.no_grad():
        for x, y, x_lengths, y_lengths in tqdm(val_loader, desc=f"Epoch {epoch} [Val]", leave=False):
            x, y, x_lengths, y_lengths = x.to(CFG.DEVICE), y.to(CFG.DEVICE), x_lengths.to(CFG.DEVICE), y_lengths.to(CFG.DEVICE)
            y_pred = model(x)
            y_pred_for_loss = y_pred.permute(1, 0, 2)
            loss = criterion(y_pred_for_loss, y, x_lengths, y_lengths)
            val_loss += loss.item() * x.size(0)
            for i in range(x.size(0)):
                pred_logits = y_pred[i, :x_lengths[i], :]
                true_indices = y[i, :y_lengths[i]]
                pred_text = greedy_decoder(pred_logits, token_map)
                true_text = decode_true_target(true_indices, token_map)
                all_pred_texts.append(pred_text)
                all_true_texts.append(true_text)
    error_rate = jiwer.wer(all_true_texts, all_pred_texts)
    return val_loss / len(val_loader.dataset), error_rate
print("Experiment setup for CTC complete.")

In [ ]:
print("Loading checkpoint weights from file...")
checkpoint = torch.load(CFG.CHECKPOINT_PATH, map_location=CFG.DEVICE, weights_only=False)
print("Checkpoint data loaded.")
for model_name, use_checkpoint in experiments_to_run:
    experiment_name = f"{model_name}_{'pretrained' if use_checkpoint else 'scratch'}"
    print(f"\n{'='*20} 🚀 STARTING EXPERIMENT: {experiment_name} {'='*20}\n")
    model = None
    if model_name == "TRANSFORMER":
        print("Initializing TransformerEncModel...")
        model = TransformerEncModel(
            data_input_size=DATA_INPUT_SIZE,
            adapter_output_size=ADAPTER_OUTPUT_SIZE,
            n_head=CFG.N_HEAD,
            num_layers=NUM_LAYERS,
            dim_feedforward=HIDDEN_SIZE,
            output_size=OUTPUT_SIZE
        )
    elif model_name in ["RNN", "LSTM", "GRU"]:
        print(f"Initializing RecurrentModel (Type: {model_name})...")
        model = RecurrentModel(
            model_type=model_name,
            data_input_size=DATA_INPUT_SIZE,
            adapter_output_size=ADAPTER_OUTPUT_SIZE,
            hidden_size=HIDDEN_SIZE,
            output_size=OUTPUT_SIZE,
            num_layers=NUM_LAYERS,
            bidirectional=IS_BIDIRECTIONAL
        )
    model = model.to(CFG.DEVICE)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total_params:,}")
    if use_checkpoint:
        print("Applying pretrained weights (strict=False)...")
        missing_keys, unexpected_keys = model.load_state_dict(
            checkpoint['model_state_dict'], 
            strict=False
        )
        print(f"  > Weights loaded. Missing keys (good): {missing_keys}")
        print(f"  > Unexpected keys (should be empty): {unexpected_keys}")
    else:
        print("Training from scratch. No checkpoint loaded.")
    criterion = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True) 
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LR)
    history = {'train_loss': [], 'val_loss': [], 'error_rate': []}
    best_error_rate = float("inf")
    best_model_path = f"/kaggle/working/best_model_{experiment_name}.pth"
    print("Starting training...")
    for epoch in range(1, CFG.EPOCHS + 1):
        train_loss = train_one_epoch(epoch, model, train_loader, criterion, optimizer)
        val_loss, error_rate = validate_one_epoch(epoch, model, val_loader, criterion, TOKEN_MAP)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['error_rate'].append(error_rate)
        print(f"Epoch {epoch}/{CFG.EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Error Rate (PER): {error_rate:.4f}")
        if error_rate < best_error_rate:
            best_error_rate = error_rate
            torch.save(model.state_dict(), best_model_path)
            print(f"✅ Saved new best model for {experiment_name} (PER: {error_rate:.4f})!")
    print(f"Running predictions for {experiment_name}...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_pred_texts = []
    all_trial_keys = []
    if len(test_loader.dataset) > 0:
        with torch.no_grad():
            for x, y, x_lengths, y_lengths, keys in tqdm(test_loader, desc=f"Testing {experiment_name}", leave=False):
                x, x_lengths = x.to(CFG.DEVICE), x_lengths.to(CFG.DEVICE)
                y_pred = model(x)
                for i in range(x.size(0)):
                    pred_logits = y_pred[i, :x_lengths[i], :] 
                    pred_text = greedy_decoder(pred_logits, TOKEN_MAP)
                    all_pred_texts.append(pred_text)
                    all_trial_keys.append(keys[i])
    print(f"Generating submission file for {experiment_name}...")
    submission_df = pd.DataFrame({'id': all_trial_keys, 'text': all_pred_texts})
    submission_df['text'] = submission_df['text'].str.strip()
    submission_path = f"/kaggle/working/submission_{experiment_name}.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"✅ Submission file saved to {submission_path}")
    all_experiment_results[experiment_name] = {
        'history': history,
        'best_val_loss': min(history['val_loss']),
        'best_error_rate': best_error_rate,
        'total_params': total_params
    }
print("\n🎉 All experiments complete! 🎉")

In [ ]:
print("📊 Experiment Results Summary\n")
sns.set(style="whitegrid", font_scale=1.1)
results_df = pd.DataFrame.from_dict(all_experiment_results, orient='index')
results_df = results_df.drop(columns='history')
print(results_df.to_markdown(floatfmt=".6f"))
model_colors = {
    "RNN": "blue",
    "LSTM": "green",
    "GRU": "red",
    "TRANSFORMER": "purple"
}
plt.figure(figsize=(16, 9)) 
for model_name, results in all_experiment_results.items():
    base_model = model_name.split('_')[0]
    is_pretrained = "pretrained" in model_name
    color = model_colors.get(base_model, 'black')
    linestyle = '-' if is_pretrained else '--'
    error_history = results['history']['error_rate']
    plt.plot(
        error_history, 
        label=f"{model_name} (Best: {results['best_error_rate']:.4f})", 
        lw=2, 
        color=color, 
        linestyle=linestyle
    )
plt.title('Model Comparison: Validation Error Rate (PER)')
plt.xlabel('Epoch')
plt.ylabel('Phoneme Error Rate (PER) (lower is better)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()
plt.figure(figsize=(16, 9))
for model_name, results in all_experiment_results.items():
    base_model = model_name.split('_')[0] 
    is_pretrained = "pretrained" in model_name
    color = model_colors.get(base_model, 'black')
    linestyle = '-' if is_pretrained else '--'
    val_loss_history = results['history']['val_loss']
    plt.plot(
        val_loss_history, 
        label=f"{model_name} (Best: {results['best_val_loss']:.4f})", 
        lw=2,
        color=color,
        linestyle=linestyle
    )
plt.title('Model Comparison: Validation Loss (CTC)')
plt.xlabel('Epoch')
plt.ylabel('Validation CTCLoss')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()
plt.figure(figsize=(16, 9))
for model_name, results in all_experiment_results.items():
    base_model = model_name.split('_')[0] 
    is_pretrained = "pretrained" in model_name
    color = model_colors.get(base_model, 'black')
    linestyle = '-' if is_pretrained else '--'
    train_loss_history = results['history']['train_loss']
    plt.plot(
        train_loss_history, 
        label=f"{model_name}", 
        lw=2,
        color=color,
        linestyle=linestyle
    )
plt.title('Model Comparison: Training Loss (CTC)')
plt.xlabel('Epoch')
plt.ylabel('Training CTCLoss')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import shutil
print("Generating final submission file...\n")
if not all_experiment_results:
    print("Warning: `all_experiment_results` is empty. Please run STEP 8 first.")
else:
    best_model_name = None
    best_model_error = float("inf")
    best_model_val_loss = float("inf")
    for model_name, results in all_experiment_results.items():
        current_error = results['best_error_rate']
        current_val_loss = results['best_val_loss']
        if current_error < best_model_error or \
           (current_error == best_model_error and current_val_loss < best_model_val_loss):
            best_model_error = current_error
            best_model_val_loss = current_val_loss
            best_model_name = model_name
    print(f"🏆 The best overall model is: {best_model_name}")
    print(f"   > Best Phoneme Error Rate (PER): {best_model_error:.6f}")
    print(f"   > Best Validation Loss: {best_model_val_loss:.6f}")
    best_submission_path = f"/kaggle/working/submission_{best_model_name}.csv"
    final_submission_path = "/kaggle/working/submission.csv"
    try:
        shutil.copyfile(best_submission_path, final_submission_path)
        print(f"\n✅ Successfully copied '{best_submission_path}' to '{final_submission_path}'")
        final_df = pd.read_csv(final_submission_path)
        print("\nFinal submission file head:")
        print(final_df.head())
    except FileNotFoundError:
        print(f"Error: Could not find '{best_submission_path}'. Make sure STEP 8 ran correctly.")
    except Exception as e:
        print(f"An error occurred while copying the file: {e}")